# **Homework 12 - Recommendation Systems**

**Context:**

Recommendation systems analyze past user behavior to identify patterns and provide personalized suggestions. In this homework, I explored several recommendation techniques and applied them to both a simple purchase history dataset and a real music listening dataset.

**Problem/Purpose:**

The purpose of this homework was to understand how recommendation systems work and how they can be built using Python. The goal was to analyze user interactions, measure similarities between users and items, and generate recommendations based on previous behavior. This helps demonstrate how businesses personalize content for their customers.

In [1]:
import pandas as pd
import numpy as np

interaction = pd.read_csv(
    'https://bradfordtuckfield.com/purchasehistory1.csv'
)

interaction.set_index("Unnamed: 0", inplace=True)

print(interaction)


            user1  user2  user3  user4  user5
Unnamed: 0                                   
item1           1      1      0      1      1
item2           1      0      1      1      0
item3           1      1      0      1      1
item4           1      0      1      0      1
item5           1      1      0      0      1


In [2]:
interaction_withcounts = interaction.copy()

interaction_withcounts.loc[:, 'counts'] = (
    interaction_withcounts.sum(axis=1)
)

interaction_withcounts = interaction_withcounts.sort_values(
    by='counts',
    ascending=False
)

print(list(interaction_withcounts.index))


['item1', 'item3', 'item2', 'item4', 'item5']


In [3]:
def popularity_based(interaction):
    interaction_withcounts = interaction.copy()

    interaction_withcounts.loc[:, 'counts'] = (
        interaction_withcounts.sum(axis=1)
    )

    sorted = interaction_withcounts.sort_values(
        by='counts',
        ascending=False
    )

    most_popular = list(sorted.index)

    return most_popular


In [4]:
print(popularity_based(interaction))


['item1', 'item3', 'item2', 'item4', 'item5']


In [5]:
print(list(interaction.loc['item1', :]))


[1, 1, 0, 1, 1]


In [13]:
def dot_product(vector1, vector2):
    thedotproduct = np.sum([
        vector1[k] * vector2[k]
        for k in range(0, len(vector1))
    ])

    return thedotproduct


In [14]:
def vector_norm(vector):
    thenorm = np.sqrt(
        dot_product(vector, vector)
    )

    return thenorm


In [15]:
def cosine_similarity(vector1, vector2):
    thedotproduct = dot_product(vector1, vector2)

    thecosine = thedotproduct / (
        vector_norm(vector1) * vector_norm(vector2)
    )

    thecosine = np.round(thecosine, 4)

    return thecosine


In [16]:
import numpy as np

item1 = interaction.loc['item1', :]
item3 = interaction.loc['item3', :]

print(cosine_similarity(item1, item3))

1.0


/tmp/ipykernel_613/630714028.py:3: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  vector1[k] * vector2[k]


In [17]:
item2 = list(interaction.loc['item2', :])
item5 = list(interaction.loc['item5', :])

print(cosine_similarity(item2, item5))


0.3333


In [18]:
ouritem = 'item1'

otherrows = [
    rowname
    for rowname in interaction.index
    if rowname != ouritem
]

otheritems = interaction.loc[otherrows, :].copy()

theitem = interaction.loc[ouritem, :]


In [19]:
similarities = []

for items in otheritems.index:
    similarities.append(
        cosine_similarity(
            theitem,
            otheritems.loc[items, :]
        )
    )

otheritems['similarities'] = similarities

recommendations = list(
    otheritems.sort_values(
        by='similarities',
        ascending=False
    ).index
)


/tmp/ipykernel_613/630714028.py:3: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  vector1[k] * vector2[k]


In [20]:
print(recommendations)


['item3', 'item5', 'item2', 'item4']


In [21]:
def get_item_recommendations(interaction, itemname):
    otherrows = [
        rowname
        for rowname in interaction.index
        if rowname != itemname
    ]

    otheritems = interaction.loc[otherrows, :].copy()

    theitem = list(interaction.loc[itemname, :])

    similarities = []

    for items in otheritems.index:
        similarities.append(
            cosine_similarity(
                theitem,
                list(otheritems.loc[items, :])
            )
        )

    otheritems['similarities'] = similarities

    return list(
        otheritems.sort_values(
            by='similarities',
            ascending=False
        ).index
    )


In [22]:
get_item_recommendations(interaction, 'item1')


['item3', 'item5', 'item2', 'item4']

In [23]:
user2 = interaction.loc[:, 'user2']
user5 = interaction.loc[:, 'user5']

print(cosine_similarity(user2, user5))


0.866


/tmp/ipykernel_613/630714028.py:3: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  vector1[k] * vector2[k]


In [24]:
user3 = interaction.loc[:, 'user3']
user5 = interaction.loc[:, 'user5']

print(cosine_similarity(user3, user5))


0.3536


/tmp/ipykernel_613/630714028.py:3: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  vector1[k] * vector2[k]


In [25]:
def get_similar_users(interaction, username):
    othercolumns = [
        columnname
        for columnname in interaction.columns
        if columnname != username
    ]

    otherusers = interaction[othercolumns].copy()

    theuser = list(interaction[username])

    similarities = []

    for users in otherusers.columns:
        similarities.append(
            cosine_similarity(
                theuser,
                list(otherusers.loc[:, users])
            )
        )

    otherusers.loc['similarities', :] = similarities

    return list(
        otherusers.sort_values(
            by='similarities',
            axis=1,
            ascending=False
        ).columns
    )


In [26]:
def get_user_recommendations(interaction, username):
    similar_users = get_similar_users(
        interaction,
        username
    )

    purchase_history = interaction[similar_users[0]]

    purchased = list(
        purchase_history.loc[
            purchase_history == 1
        ].index
    )

    purchased2 = list(
        interaction.loc[
            interaction[username] == 1, :
        ].index
    )

    recs = sorted(
        list(set(purchased) - set(purchased2))
    )

    return recs


In [27]:
get_user_recommendations(interaction, 'user2')


['item4']

In [28]:
import pandas as pd

lastfm = pd.read_csv(
    "https://bradfordtuckfield.com/lastfm-matrix-germany.csv"
)

print(lastfm.head())


   user  a perfect circle  abba  ac/dc  adam green  aerosmith  afi  air  \
0     1                 0     0      0           0          0    0    0   
1    33                 0     0      0           1          0    0    0   
2    42                 0     0      0           0          0    0    0   
3    51                 0     0      0           0          0    0    0   
4    62                 0     0      0           0          0    0    0   

   alanis morissette  alexisonfire  ...  timbaland  tom waits  tool  \
0                  0             0  ...          0          0     0   
1                  0             0  ...          0          0     0   
2                  0             0  ...          0          0     0   
3                  0             0  ...          0          0     0   
4                  0             0  ...          0          0     0   

   tori amos  travis  trivium  u2  underoath  volbeat  yann tiersen  
0          0       0        0   0          0        

In [29]:
lastfm.drop(
    ['user'],
    axis=1,
    inplace=True
)


In [30]:
lastfmt = lastfm.T


In [31]:
print(lastfmt.shape)


(285, 1257)


In [32]:
get_item_recommendations(
    lastfmt,
    'abba'
)[0:10]


['madonna',
 'robbie williams',
 'elvis presley',
 'michael jackson',
 'queen',
 'the beatles',
 'kelly clarkson',
 'groove coverage',
 'duffy',
 'mika']

In [33]:
print(
    get_user_recommendations(
        lastfmt,
        0
    )[0:3]
)


/tmp/ipykernel_613/3435528478.py:4: RuntimeWarning: invalid value encountered in scalar divide
  thecosine = thedotproduct / (


['billy talent', 'bob marley', 'die toten hosen']


**Solution/Approach:**

I began by examining a purchase history dataset and identifying the most popular items. Next, I used mathematical concepts such as dot products, vector norms, and cosine similarity to compare items and users. These calculations allowed me to create both item-based and user-based collaborative filtering recommendation systems.

After developing the recommendation functions, I applied them to a Last.fm music dataset. The recommendation system identified artists with similar listening patterns and generated personalized music recommendations for individual users.

**Results:**

The recommendation models successfully produced ranked suggestions based on user interactions and similarity measurements. The popularity-based method identified the most frequently purchased items, while collaborative filtering generated more personalized recommendations. The music dataset also produced reasonable artist recommendations based on listening behavior, demonstrating how these techniques can be used in real-world applications.

**Conclusion:**

This provided me with hands-on experience with recommendation systems and collaborative filtering techniques. By analyzing user behavior and measuring similarities between users and items, I was able to create recommendation models that generate personalized suggestions.